# Лабораторная работа №5: История и два агента

[Исходное задание](https://colab.research.google.com/drive/1n_H7xNzo2irkW0buNw_EQDvOXEZcTKCE)

Выполнение через совместимый API RouterAI. API-ключ берётся из окружения или локального `.env`. Сохранённые результаты получены реальными запросами.



## Задание

#Задание семинара 5
1. Реализуйте диалог с LLM с историей ограниченной длины (например, длиной 6), при превышении длины происходит удаление из начала старых запросов и ответов при условии, что их длина была больше 100 символов.
2. Реализуйте диалог между двумя виртуальными агентами до тех пор, пока один из агентов не выдаст ответ, состоящий из менее, чем 10 символов.

## Подключение

In [ ]:
from pathlib import Path
import sys

# При открытии из подпапки находим общий модуль конфигурации.
for candidate in [Path.cwd(), Path.cwd().parent]:
    if (candidate / "lab_runtime.py").exists():
        sys.path.insert(0, str(candidate))
        break

from lab_runtime import make_llm, json_result, ASSETS
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from IPython.display import display, Markdown, Image, Audio

llm = make_llm()
print("Модель:", llm.model_name)


## 1. История длиной 6 с удалением длинных сообщений

Системная роль хранится отдельно. Удаляем старые сообщения длиннее 100 символов. Если подходящих сообщений нет, сообщаем о конфликте ограничений: одновременно сохранить все короткие сообщения и удержать размер 6 невозможно.

In [ ]:
MAX_HISTORY = 6
history = []
removed = []

def trim_history(history, limit=MAX_HISTORY):
    while len(history) > limit:
        eligible = next((i for i, message in enumerate(history) if len(message.content) > 100), None)
        if eligible is None:
            raise ValueError("История переполнена, но сообщений длиннее 100 символов для удаления нет")
        old = history.pop(eligible)
        removed.append({"role": old.type, "length": len(old.content)})

system = SystemMessage(content="Ты помощник по Python. Отвечай в 2-3 предложениях, примерно 150 символов.")
topics = ["списки", "словари", "функции", "циклы", "генераторы"]
for topic in topics:
    user_text = f"Объясни тему {topic} в Python. Приведи пример применения этой возможности при обработке текстов песен, поясни преимущество и возможное ограничение."
    history.append(HumanMessage(content=user_text))
    trim_history(history)
    reply = llm.invoke([system] + history)
    history.append(reply)
    trim_history(history)
    print("Тема:", topic, "Размер истории:", len(history))
    print(reply.content)
print("Удалены:", removed)
assert len(history) <= MAX_HISTORY
assert all(item["length"] > 100 for item in removed)


## 2. Диалог двух виртуальных агентов

Один предлагает архитектуру, другой проверяет её. Завершение — ответ любого агента короче 10 символов. Лимит раундов защищает от бесконечного диалога.

In [ ]:
agent_a = make_llm(max_tokens=250)
agent_b = make_llm(max_tokens=250)
histories = [
    [SystemMessage(content="Ты архитектор. Предложи простое решение проверки текстов песен. Обсуждай замечания рецензента. Если согласен завершить, ответь ровно: Готово.")],
    [SystemMessage(content="Ты рецензент. Проверь решение архитектора, дай максимум одно замечание. Если решение достаточно хорошее, ответь ровно: ОК.")],
]
models = [agent_a, agent_b]
incoming = "Предложи архитектуру: пользователь отправляет текст, система проверяет ключевые слова и выдаёт результат."
transcript = []
finished = False
for turn in range(12):
    index = turn % 2
    histories[index].append(HumanMessage(content=incoming))
    answer = models[index].invoke(histories[index])
    histories[index].append(answer)
    text = answer.content.strip()
    transcript.append((index + 1, text))
    print(f"Агент {index + 1} ({len(text)} символов): {text}\n")
    if len(text) < 10:
        finished = True
        print("Стоп: ответ короче 10 символов.")
        break
    incoming = text
assert finished, "За 12 реплик агенты не завершили обсуждение"


## Вывод

Реализованы фильтрация переполненной истории по длине сообщения и автоматический обмен репликами двух агентов с проверкой условия остановки.